Unzip the zip folder

In [7]:
from google.colab import files
import zipfile, os

# Upload your zip file
uploaded = files.upload()

# Get the uploaded filename
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

# Extract ZIP into a folder called 'data'
with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall('data')

# Check folder structure
for root, dirs, files in os.walk('data'):
    level = root.replace('data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")


Saving OOP Annotated.zip to OOP Annotated.zip
Uploaded file: OOP Annotated.zip
data/
  OOP Annotated/
    VU/
      spacy_train_data.py
      OOP-VU.docx
      VU.txt
      annotations (1).json
      tags.json
    UET/
      spacy_train_data.py
      annotations (1).json
      tags.json
      OOP-UET.docx
    Riphah/
      spacy_train_data.py
      annotations (1).json
      Riphah.txt
      tags.json
      Riphah-OOP.docx
    Bahria/
      spacy_train_data.py
      tags (1).json
      OOP-Bahria.docx
      annotations (2).json
    UMT/
      spacy_train_data.py
      annotations (1).json
      tags.json
      OOP-UMT.docx
    Adelaide/
      spacy_train_data.py
      OOP-Adelaide.docx
      annotations (1).json
      tags.json
      OOP-Adelaide.txt
    Spatial_Umaine/
      spacy_train_data.py
      annotations (1).json
      Spatial_Umaine.txt
      tags.json
      OOP-Spatial_Umaine.docx
    UOL/
      spacy_train_data.py
      annotations (1).json
      OOP-UOL.docx
      tags.jso

Scibert

In [8]:
# ========================
# STEP 1: Install Packages
# ========================
!pip install -U spacy spacy-transformers torch

import os, random, numpy as np, torch, importlib.util
import spacy
import spacy_transformers
from spacy.training.example import Example
from spacy.util import minibatch

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ========================
# STEP 2: Unzip Your Data
# ========================
!unzip -o "OOP Annotated (1).zip" -d /content/data

base_path = "/content/data/OOP Annotated"

# ========================
# STEP 3: Collect TRAIN_DATA from all spacy_train_data.py
# ========================
TRAIN_DATA = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file == "spacy_train_data.py":
            file_path = os.path.join(root, file)
            print(f"📂 Loading: {file_path}")

            # Import spacy_train_data.py dynamically
            spec = importlib.util.spec_from_file_location("spacy_train_data", file_path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)

            TRAIN_DATA.extend(module.TRAIN_DATA)

print(f"✅ Collected {len(TRAIN_DATA)} samples from all subfolders")

# ========================
# STEP 4: Train/Validation Split (90/10)
# ========================
random.shuffle(TRAIN_DATA)
train_size = int(0.9 * len(TRAIN_DATA))

train_data = TRAIN_DATA[:train_size]
dev_data = TRAIN_DATA[train_size:]

print(f"Train: {len(train_data)}, Validation: {len(dev_data)}")

# ========================
# STEP 5: Load Transformer (SciBERT)
# ========================
transformer_model = "allenai/scibert_scivocab_uncased"  # or "microsoft/codebert-base"

nlp = spacy.blank("en")
transformer = nlp.add_pipe("transformer", config={"model": {"name": transformer_model}})
ner = nlp.add_pipe("ner")

# Add labels
for _, ann in TRAIN_DATA:
    for ent in ann["entities"]:
        ner.add_label(ent[2])

# ========================
# STEP 6: Evaluation Function
# ========================
def evaluate(nlp, data):
    tp = fp = fn = 0
    for text, annotations in data:
        doc = nlp(text)
        gold_entities = {(s, e, l) for s, e, l in annotations["entities"]}
        pred_entities = {(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents}
        tp += len(gold_entities & pred_entities)
        fp += len(pred_entities - gold_entities)
        fn += len(gold_entities - pred_entities)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return {"p": precision, "r": recall, "f1": f1}

# ========================
# STEP 7: Training Loop
# ========================
optimizer = nlp.initialize()
best_f1 = 0.0

for epoch in range(15):  # adjust epochs if needed
    losses = {}
    random.shuffle(train_data)
    batches = minibatch(train_data, size=8)
    for batch in batches:
        examples = []
        for text, annotations in batch:
            doc = nlp.make_doc(text)
            examples.append(Example.from_dict(doc, annotations))
        nlp.update(examples, sgd=optimizer, losses=losses)

    scores = evaluate(nlp, dev_data)
    print(f"Epoch {epoch} Losses: {losses} Scores: {scores}")

    if scores["f1"] > best_f1:
        best_f1 = scores["f1"]
        nlp.to_disk("/content/cs_ner_transformer_best")
        print(f"✅ New best model saved with F1 = {best_f1:.3f}")

# ========================
# STEP 8: Final Validation Evaluation
# ========================
best_model = spacy.load("/content/cs_ner_transformer_best")
val_scores = evaluate(best_model, dev_data)
print("🎯 Final Validation Scores:", val_scores)


unzip:  cannot find or open OOP Annotated (1).zip, OOP Annotated (1).zip.zip or OOP Annotated (1).zip.ZIP.
📂 Loading: /content/data/OOP Annotated/VU/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UET/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Riphah/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Bahria/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UMT/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Adelaide/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Spatial_Umaine/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UOL/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Florida State/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/GIKI/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/IBA/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UCP/spacy_train_data.py
✅ Collected 309 samples from all subfolders
Train: 278, Validation: 31
Epoch 0 Losses: {'transformer': 0.

CodeBert

In [9]:
# ========================
# STEP 1: Install Packages
# ========================
!pip install -U spacy spacy-transformers torch

import os, random, numpy as np, torch, importlib.util
import spacy
from spacy.training.example import Example
from spacy.util import minibatch

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ========================
# STEP 2: Unzip Your Data
# ========================
!unzip -o "OOP Annotated (1).zip" -d /content/data

base_path = "/content/data/OOP Annotated"

# ========================
# STEP 3: Collect TRAIN_DATA from all spacy_train_data.py
# ========================
TRAIN_DATA = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file == "spacy_train_data.py":
            file_path = os.path.join(root, file)
            print(f"📂 Loading: {file_path}")

            # Import spacy_train_data.py dynamically
            spec = importlib.util.spec_from_file_location("spacy_train_data", file_path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)

            TRAIN_DATA.extend(module.TRAIN_DATA)

print(f"✅ Collected {len(TRAIN_DATA)} samples from all subfolders")

# ========================
# STEP 4: Train/Validation Split (90/10)
# ========================
random.shuffle(TRAIN_DATA)
train_size = int(0.9 * len(TRAIN_DATA))

train_data = TRAIN_DATA[:train_size]
dev_data = TRAIN_DATA[train_size:]

print(f"Train: {len(train_data)}, Validation: {len(dev_data)}")

# ========================
# STEP 5: Load Transformer (CodeBERT)
# ========================
transformer_model = "microsoft/codebert-base"  # 🔹 CodeBERT

nlp = spacy.blank("en")
transformer = nlp.add_pipe("transformer", config={"model": {"name": transformer_model}})
ner = nlp.add_pipe("ner")

# Add labels
for _, ann in TRAIN_DATA:
    for ent in ann["entities"]:
        ner.add_label(ent[2])

# ========================
# STEP 6: Evaluation Function
# ========================
def evaluate(nlp, data):
    tp = fp = fn = 0
    for text, annotations in data:
        doc = nlp(text)
        gold_entities = {(s, e, l) for s, e, l in annotations["entities"]}
        pred_entities = {(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents}
        tp += len(gold_entities & pred_entities)
        fp += len(pred_entities - gold_entities)
        fn += len(gold_entities - pred_entities)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return {"p": precision, "r": recall, "f1": f1}

# ========================
# STEP 7: Training Loop
# ========================
optimizer = nlp.initialize()
best_f1 = 0.0

for epoch in range(15):  # adjust epochs if needed
    losses = {}
    random.shuffle(train_data)
    batches = minibatch(train_data, size=8)
    for batch in batches:
        examples = []
        for text, annotations in batch:
            doc = nlp.make_doc(text)
            examples.append(Example.from_dict(doc, annotations))
        nlp.update(examples, sgd=optimizer, losses=losses)

    scores = evaluate(nlp, dev_data)
    print(f"Epoch {epoch} Losses: {losses} Scores: {scores}")

    if scores["f1"] > best_f1:
        best_f1 = scores["f1"]
        nlp.to_disk("/content/cs_ner_codebert_best")  # 🔹 Save separately for CodeBERT
        print(f"✅ New best CodeBERT model saved with F1 = {best_f1:.3f}")

# ========================
# STEP 8: Final Validation Evaluation
# ========================
best_model = spacy.load("/content/cs_ner_codebert_best")
val_scores = evaluate(best_model, dev_data)
print("🎯 Final Validation Scores (CodeBERT):", val_scores)


unzip:  cannot find or open OOP Annotated (1).zip, OOP Annotated (1).zip.zip or OOP Annotated (1).zip.ZIP.
📂 Loading: /content/data/OOP Annotated/VU/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UET/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Riphah/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Bahria/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UMT/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Adelaide/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Spatial_Umaine/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UOL/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Florida State/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/GIKI/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/IBA/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UCP/spacy_train_data.py
✅ Collected 309 samples from all subfolders
Train: 278, Validation: 31


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Epoch 0 Losses: {'transformer': 0.0, 'ner': np.float32(1107.4855)} Scores: {'p': 0.2941176470011534, 'r': 0.25862068961058265, 'f1': 0.2752293527682856}
✅ New best CodeBERT model saved with F1 = 0.275
Epoch 1 Losses: {'transformer': 0.0, 'ner': np.float32(589.95886)} Scores: {'p': 0.5172413792211653, 'r': 0.5172413792211653, 'f1': 0.5172413742211655}
✅ New best CodeBERT model saved with F1 = 0.517
Epoch 2 Losses: {'transformer': 0.0, 'ner': np.float32(312.91995)} Scores: {'p': 0.7407407406035665, 'r': 0.689655172294887, 'f1': 0.7142857091645409}
✅ New best CodeBERT model saved with F1 = 0.714
Epoch 3 Losses: {'transformer': 0.0, 'ner': np.float32(306.55038)} Scores: {'p': 0.6666666665497076, 'r': 0.6551724136801427, 'f1': 0.6608695601028356}
Epoch 4 Losses: {'transformer': 0.0, 'ner': np.float32(233.51633)} Scores: {'p': 0.642857142742347, 'r': 0.6206896550653983, 'f1': 0.6315789422591568}
Epoch 5 Losses: {'transformer': 0.0, 'ner': np.float32(235.57887)} Scores: {'p': 0.64912280690366

rule-base

In [10]:
import os
import spacy
import importlib.util

# ======================
# 1. Traverse All Folders and Collect TRAIN_DATA
# ======================
base_path = "/content/data/OOP Annotated"

TRAIN_DATA = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file == "spacy_train_data.py":
            file_path = os.path.join(root, file)
            print(f"📂 Loading: {file_path}")

            # Dynamically import
            spec = importlib.util.spec_from_file_location("spacy_train_data", file_path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)

            TRAIN_DATA.extend(module.TRAIN_DATA)

print(f"✅ Collected {len(TRAIN_DATA)} samples from all subfolders")

# ======================
# 2. Extract Unique Entity Patterns with Lemmatization
# ======================
nlp_temp = spacy.load("en_core_web_sm")  # use spaCy's small model for lemmatization
patterns = []
seen = set()

for text, ann in TRAIN_DATA:
    for start, end, label in ann["entities"]:
        span_text = text[start:end].strip()
        if span_text:
            # Lemmatize entity text
            doc = nlp_temp(span_text)
            lemma_text = " ".join([token.lemma_ for token in doc])

            # Save both original and lemmatized versions
            for variant in {span_text, lemma_text}:
                key = (variant.lower(), label)
                if key not in seen:
                    seen.add(key)
                    patterns.append({"label": label, "pattern": variant})

print(f"✅ Extracted {len(patterns)} unique patterns (with lemma variants) for rules")

# ======================
# 3. Build Rule-Based NER
# ======================
nlp_rules = spacy.blank("en")
ruler = nlp_rules.add_pipe("entity_ruler", config={"overwrite_ents": True})
ruler.add_patterns(patterns)

# Save rule-based model
save_path = "/content/rule_based_ner"
nlp_rules.to_disk(save_path)
print(f"✅ Rule-based NER saved at {save_path}")

# ======================
# 4. Quick Test
# ======================
test_texts = [
    "Virtual Destructors are useful.",
    "A Virtual Destructor is important in OOP.",
    "Destructor handling is tricky."
]

for text in test_texts:
    doc = nlp_rules(text)
    print(f"\n🔎 Text: {text}")
    for ent in doc.ents:
        print(f" → {ent.text} [{ent.label_}]")


📂 Loading: /content/data/OOP Annotated/VU/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UET/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Riphah/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Bahria/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UMT/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Adelaide/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Spatial_Umaine/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UOL/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Florida State/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/GIKI/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/IBA/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UCP/spacy_train_data.py
✅ Collected 309 samples from all subfolders
✅ Extracted 637 unique patterns (with lemma variants) for rules
✅ Rule-based NER saved at /content/rule_based_ner

🔎 Text: Virtual Destructors are useful.
 → Virtual De

ensemble( scibert + codebert + rulebase)

In [16]:
import spacy
from spacy.language import Language
from spacy.tokens import Doc, Span
from rapidfuzz import fuzz
from collections import Counter

# ======================
# Load Base Models
# ======================
print("📂 Loading base models...")
scibert = spacy.load("/content/cs_ner_transformer_best")
codebert = spacy.load("/content/cs_ner_codebert_best")
rulebase = spacy.load("/content/rule_based_ner")
nlp_temp = spacy.load("en_core_web_sm")  # for lemmatization
print("✅ Loaded SciBERT, CodeBERT, Rule-based")

# ======================
# IoU Function
# ======================
def span_iou(span1, span2):
    inter_start = max(span1.start, span2.start)
    inter_end = min(span1.end, span2.end)
    intersection = max(0, inter_end - inter_start)
    union = max(span1.end, span2.end) - min(span1.start, span2.start)
    return intersection / union if union > 0 else 0

# ======================
# Lemmatization helper
# ======================
def get_lemma(text: str) -> str:
    doc = nlp_temp(text)
    return " ".join([token.lemma_ for token in doc])

# ======================
# Filter overlapping spans
# ======================
def filter_overlapping_spans(spans):
    """Keep non-overlapping spans, prefer longer ones if overlap."""
    spans = sorted(spans, key=lambda s: (s.start, -s.end))
    result, seen_tokens = [], set()
    for span in spans:
        if any(t in seen_tokens for t in range(span.start, span.end)):
            continue
        result.append(span)
        seen_tokens.update(range(span.start, span.end))
    return result

# ======================
# Ensemble Factory
# ======================
@Language.factory("ensemble_ner_hybrid")
def create_ensemble_ner_hybrid(nlp, name, fuzzy_threshold=85):
    """
    Hybrid Ensemble:
    1. Always keep rule-based entities (conf=1.0)
    2. Keep SciBERT ∩ CodeBERT agreement (conf=0.9)
    3. Keep Rule-based ∩ Transformer (conf=1.0)
    4. Allow single transformer entities if:
       - Lemma matches a rule entity, OR
       - Fuzzy similarity ≥ threshold
    """
    decision_stats = Counter()

    def ensemble_component(doc: Doc):
        docs = [scibert(doc.text), codebert(doc.text), rulebase(doc.text)]
        ents, kept_spans = [], set()

        # Collect rule-based lemmas for reference
        rule_lemmas = {get_lemma(ent.text).lower(): ent.label_ for ent in docs[2].ents}

        # --- Always keep rule-based entities ---
        for ent in docs[2].ents:
            span = Span(doc, ent.start, ent.end, label=ent.label_)
            span._.confidence = 1.0
            span._.decision = "Rule"
            ents.append(span)
            kept_spans.add((span.start, span.end, span.label_))

        # --- Transformer agreement (SciBERT ∩ CodeBERT) ---
        for ent1 in docs[0].ents:
            for ent2 in docs[1].ents:
                if ent1.label_ == ent2.label_ and span_iou(ent1, ent2) >= 0.5:
                    span = Span(doc, ent1.start, ent1.end, label=ent1.label_)
                    if (span.start, span.end, span.label_) not in kept_spans:
                        span._.confidence = 0.9
                        span._.decision = "SciBERT+CodeBERT"
                        ents.append(span)
                        kept_spans.add((span.start, span.end, span.label_))

        # --- Rule ∩ Transformer agreement ---
        for rule_ent in docs[2].ents:
            for model_doc in [docs[0], docs[1]]:
                for ent in model_doc.ents:
                    if ent.label_ == rule_ent.label_ and span_iou(ent, rule_ent) >= 0.5:
                        span = Span(doc, rule_ent.start, rule_ent.end, label=rule_ent.label_)
                        if (span.start, span.end, span.label_) not in kept_spans:
                            span._.confidence = 1.0
                            span._.decision = "Rule+Transformer"
                            ents.append(span)
                            kept_spans.add((span.start, span.end, span.label_))

        # --- Single Transformer Fallback (lemma + fuzzy) ---
        for model_name, model_doc in [("SciBERT", docs[0]), ("CodeBERT", docs[1])]:
            for ent in model_doc.ents:
                span = Span(doc, ent.start, ent.end, label=ent.label_)
                lemma = get_lemma(ent.text).lower()

                # Lemma match with rule
                if lemma in rule_lemmas and rule_lemmas[lemma] == ent.label_:
                    if (span.start, span.end, span.label_) not in kept_spans:
                        span._.confidence = 0.75
                        span._.decision = f"{model_name}+LemmaMatch"
                        ents.append(span)
                        kept_spans.add((span.start, span.end, span.label_))
                        continue

                # Fuzzy match with rule
                for rule_text, rule_label in rule_lemmas.items():
                    if rule_label == ent.label_ and fuzz.ratio(lemma, rule_text) >= fuzzy_threshold:
                        if (span.start, span.end, span.label_) not in kept_spans:
                            span._.confidence = 0.7
                            span._.decision = f"{model_name}+Fuzzy"
                            ents.append(span)
                            kept_spans.add((span.start, span.end, span.label_))
                        break

        # --- Filter overlaps ---
        doc.ents = filter_overlapping_spans(ents)
        return doc

    return ensemble_component

# ======================
# Register extensions
# ======================
if not Span.has_extension("confidence"):
    Span.set_extension("confidence", default=0.0)
if not Span.has_extension("decision"):
    Span.set_extension("decision", default="Unknown")

# ======================
# Build NLP pipeline
# ======================
nlp = spacy.blank("en")
nlp.add_pipe("ensemble_ner_hybrid", config={"fuzzy_threshold": 85})
print("✅ Hybrid Ensemble (Rules + Transformers + Lemma/Fuzzy) loaded")

# ======================
# Quick Test
# ======================
doc = nlp("Virtual Destructors and Polymorphism are OOP concepts. Destructor handling is tricky.")
for ent in doc.ents:
    print(ent.text, ent.label_, f"(conf={ent._.confidence:.2f}, src={ent._.decision})")


📂 Loading base models...
✅ Loaded SciBERT, CodeBERT, Rule-based
✅ Hybrid Ensemble (Rules + Transformers + Lemma/Fuzzy) loaded
Virtual Destructors CONCEPT (conf=1.00, src=Rule)
Polymorphism CONCEPT (conf=0.90, src=SciBERT+CodeBERT)
Destructor handling is tricky. CONCEPT (conf=0.90, src=SciBERT+CodeBERT)


clear colab RAM

In [ ]:
# Clear all variables
%reset -f

# Clear cache (helpful if using PyTorch)
import gc
gc.collect()

# If using GPU with PyTorch
try:
    import torch
    torch.cuda.empty_cache()
except:
    pass


save ensemble model

In [17]:
nlp.to_disk("/content/ensemble_ner_hybrid")
print("📂 Saved to /content/ensemble_ner_hybrid")


📂 Saved to /content/ensemble_ner_hybrid


NER by Ensemble

In [18]:
!pip install python-docx

import os, json, spacy
from docx import Document

# ==========================
# STEP 1: Path to Your Folder
# ==========================
folder_path = "/content/data/OOP Annotated"

# ==========================
# STEP 2: Load your model
# ==========================
print("📂 Loading Ensemble NER model...")
nlp = spacy.load("/content/ensemble_ner_hybrid")  # <-- adjust to your model path
print("✅ Model loaded successfully!")

# ==========================
# STEP 3: Read DOCX Function (returns list of paragraphs)
# ==========================
def read_docx_paragraphs(file_path):
    doc = Document(file_path)
    return [p.text.strip() for p in doc.paragraphs if p.text.strip()]

# ==========================
# STEP 4: Helper to filter overlapping spans
# ==========================
def filter_overlapping_spans(spans):
    """Keep non-overlapping spans, prefer longer ones if overlap."""
    spans = sorted(spans, key=lambda s: (s.start, -s.end))  # sort by start, then longer first
    result = []
    seen_tokens = set()
    for span in spans:
        if any(t in seen_tokens for t in range(span.start, span.end)):
            continue  # skip overlapping
        result.append(span)
        seen_tokens.update(range(span.start, span.end))
    return result

# ==========================
# STEP 5: Process All Word Files (grouped by paragraph)
# ==========================
all_results = {}

for root, dirs, files_list in os.walk(folder_path):
    for file in files_list:
        if file.endswith(".docx") and not file.startswith("~$"):  # ✅ Skip temp files
            file_path = os.path.join(root, file)
            print(f"🔎 Processing {file} ...")

            paragraphs = read_docx_paragraphs(file_path)
            file_results = []

            for para in paragraphs:
                doc = nlp(para)

                # filter overlapping entities
                ents = []
                non_overlapping = filter_overlapping_spans(list(doc.ents))
                for ent in non_overlapping:
                    ents.append({
                        "text": ent.text,
                        "label": ent.label_,
                        "confidence": float(ent._.confidence)
                    })

                if ents:  # only add paragraph if entities found
                    file_results.append({
                        "paragraph": para,
                        "entities": ents
                    })

            all_results[file] = file_results

# ==========================
# STEP 6: Save Results
# ==========================
output_file = "/content/all_ner_results_by_paragraph.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False)

print(f"✅ Results saved to {output_file}")


📂 Loading Ensemble NER model...
✅ Model loaded successfully!
🔎 Processing OOP-VU.docx ...
🔎 Processing OOP-UET.docx ...
🔎 Processing Riphah-OOP.docx ...
🔎 Processing OOP-Bahria.docx ...
🔎 Processing OOP-UMT.docx ...
🔎 Processing OOP-Adelaide.docx ...
🔎 Processing OOP-Spatial_Umaine.docx ...
🔎 Processing OOP-UOL.docx ...
🔎 Processing OOP-Florida State University.docx ...
🔎 Processing OOP-McMaster.docx ...
🔎 Processing OOP-GIKI.docx ...
🔎 Processing OOP-IBA.docx ...
🔎 Processing OOP-UCP.docx ...
✅ Results saved to /content/all_ner_results_by_paragraph.json


comparison of ner and annotated

In [19]:
import os
import importlib.util
import json
from sklearn.metrics import classification_report, precision_recall_fscore_support
import spacy

# ==========================
# STEP 1: Load predictions
# ==========================
pred_file = "/content/all_ner_results_by_paragraph.json"
with open(pred_file, "r", encoding="utf-8") as f:
    pred_data = json.load(f)

# ==========================
# STEP 2: Load gold training data
# ==========================
def load_spacy_train_data(py_file):
    spec = importlib.util.spec_from_file_location("spacy_train_data", py_file)
    spacy_data = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(spacy_data)
    return spacy_data.TRAIN_DATA  # [(text, {"entities": [...]})]

# ==========================
# STEP 3: Helpers
# ==========================
def normalize_label(label):
    if label in ["OTHER", "OTHERS"]:
        return "OTHER"
    return label

def span_iou(span1, span2):
    """IoU over character spans"""
    start1, end1 = span1
    start2, end2 = span2
    inter_start = max(start1, start2)
    inter_end = min(end1, end2)
    intersection = max(0, inter_end - inter_start)
    union = max(end1, end2) - min(start1, start2)
    return intersection / union if union > 0 else 0

def evaluate_one_doc(gold_data, pred_file_data, iou_thresh=0.5):
    gold_labels, pred_labels = [], []

    # Convert preds into spans with offsets
    pred_spans = []
    for para in pred_file_data:
        para_text = para["paragraph"]
        for ent in para["entities"]:
            ent_text = ent["text"]
            label = normalize_label(ent["label"])
            start = para_text.lower().find(ent_text.lower())
            if start != -1:
                pred_spans.append((start, start+len(ent_text), label))

    # Convert gold into spans
    gold_spans = [(start, end, normalize_label(label))
                  for text, ann in gold_data
                  for start, end, label in ann["entities"]]

    matched_pred = set()

    # Check gold vs predictions
    for g_start, g_end, g_label in gold_spans:
        gold_labels.append(g_label)
        found_match = False
        for idx, (p_start, p_end, p_label) in enumerate(pred_spans):
            if p_label == g_label and span_iou((g_start,g_end),(p_start,p_end)) >= iou_thresh:
                pred_labels.append(p_label)
                matched_pred.add(idx)
                found_match = True
                break
        if not found_match:
            pred_labels.append("MISS")

    # Add false positives
    for idx, (p_start, p_end, p_label) in enumerate(pred_spans):
        if idx not in matched_pred:
            gold_labels.append("NONE")
            pred_labels.append(p_label)

    return gold_labels, pred_labels

# ==========================
# STEP 4: Loop through folders
# ==========================
root_folder = "/content/data/OOP Annotated"

all_gold, all_pred = [], []
per_folder_results = {}

for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file == "spacy_train_data.py":
            gold_file = os.path.join(root, file)
            gold_data = load_spacy_train_data(gold_file)

            folder_name = os.path.basename(root)
            docx_name = f"OOP-{folder_name}.docx"

            pred_file_data = pred_data.get(docx_name, [])

            g_labels, p_labels = evaluate_one_doc(gold_data, pred_file_data)

            all_gold.extend(g_labels)
            all_pred.extend(p_labels)

            if g_labels:
                labels = sorted(set([l for l in g_labels if l not in ["MISS","NONE"]]))
                if labels:
                    report = classification_report(
                        g_labels, p_labels,
                        labels=labels,
                        digits=4,
                        output_dict=True
                    )
                    per_folder_results[folder_name] = report["weighted avg"]

# ==========================
# STEP 5: Global Evaluation
# ==========================
print("📊 Global Classification Report (IoU-based, ignoring MISS/NONE)")
labels = sorted(set([l for l in all_gold if l not in ["MISS","NONE"]]))
print(classification_report(all_gold, all_pred, labels=labels, digits=4))

precision, recall, f1, _ = precision_recall_fscore_support(
    all_gold, all_pred, labels=labels, average="weighted"
)

print(f"\nOverall Precision: {precision:.4f}")
print(f"Overall Recall:    {recall:.4f}")
print(f"Overall F1-score:  {f1:.4f}")

# ==========================
# STEP 6: Per-folder summary
# ==========================
print("\n📂 Per-folder weighted F1-scores:")
for folder, scores in per_folder_results.items():
    print(f"{folder:30s}  P={scores['precision']:.4f}  R={scores['recall']:.4f}  F1={scores['f1-score']:.4f}")


📊 Global Classification Report (IoU-based, ignoring MISS/NONE)
              precision    recall  f1-score   support

     CONCEPT     0.5992    0.3407    0.4344       452
      METHOD     0.0000    0.0000    0.0000         1
       OTHER     0.6136    0.8182    0.7013        33
   TECHNIQUE     0.5000    1.0000    0.6667         2
       TOPIC     0.8500    0.2537    0.3908        67

   micro avg     0.6154    0.3604    0.4545       555
   macro avg     0.5126    0.4825    0.4386       555
weighted avg     0.6289    0.3604    0.4451       555


Overall Precision: 0.6289
Overall Recall:    0.3604
Overall F1-score:  0.4451

📂 Per-folder weighted F1-scores:
VU                              P=0.5928  R=0.9767  F1=0.7347
UET                             P=0.0000  R=0.0000  F1=0.0000
Riphah                          P=0.0000  R=0.0000  F1=0.0000
Bahria                          P=0.5962  R=1.0000  F1=0.7375
UMT                             P=0.0000  R=0.0000  F1=0.0000
Adelaide                 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

rule ner

In [20]:
import os, json, spacy, importlib.util
from spacy.pipeline import EntityRuler
from docx import Document

# ==========================
# STEP 1: Collect TRAIN_DATA from spacy_train_data.py
# ==========================
base_path = "/content/data/OOP Annotated"
TRAIN_DATA = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file == "spacy_train_data.py":
            file_path = os.path.join(root, file)
            print(f"📂 Loading: {file_path}")

            spec = importlib.util.spec_from_file_location("spacy_train_data", file_path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)

            TRAIN_DATA.extend(module.TRAIN_DATA)

print(f"✅ Collected {len(TRAIN_DATA)} samples from all subfolders")

# ==========================
# STEP 2: Build spaCy pipeline with EntityRuler
# ==========================
nlp_rule = spacy.blank("en")
ruler = nlp_rule.add_pipe("entity_ruler")

patterns = []
for text, ann in TRAIN_DATA:
    for start, end, label in ann["entities"]:
        span_text = text[start:end]
        patterns.append({"label": label, "pattern": span_text})

ruler.add_patterns(patterns)
print(f"✅ Added {len(patterns)} rule-based patterns from TRAIN_DATA")

# ==========================
# STEP 3: Read DOCX Function
# ==========================
def read_docx_paragraphs(file_path):
    doc = Document(file_path)
    return [p.text.strip() for p in doc.paragraphs if p.text.strip()]

# ==========================
# STEP 4: Process All Word Files (Rule-Based NER)
# ==========================
rule_results = {}

for root, dirs, files_list in os.walk(base_path):
    for file in files_list:
        if file.endswith(".docx") and not file.startswith("~$"):
            file_path = os.path.join(root, file)
            print(f"🔎 Rule-based processing {file} ...")

            paragraphs = read_docx_paragraphs(file_path)
            file_results = []

            for para in paragraphs:
                doc = nlp_rule(para)
                ents = [{"text": ent.text, "label": ent.label_} for ent in doc.ents]

                if ents:
                    file_results.append({
                        "paragraph": para,
                        "entities": ents
                    })
            rule_results[file] = file_results

# ==========================
# STEP 5: Save Rule-Based Results
# ==========================
output_file = "/content/rule_based_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(rule_results, f, indent=4, ensure_ascii=False)

print(f"✅ Rule-based NER results saved to {output_file}")


📂 Loading: /content/data/OOP Annotated/VU/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UET/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Riphah/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Bahria/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UMT/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Adelaide/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Spatial_Umaine/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UOL/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/Florida State/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/GIKI/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/IBA/spacy_train_data.py
📂 Loading: /content/data/OOP Annotated/UCP/spacy_train_data.py
✅ Collected 309 samples from all subfolders
✅ Added 555 rule-based patterns from TRAIN_DATA
🔎 Rule-based processing OOP-VU.docx ...
🔎 Rule-based processing OOP-UET.docx ...
🔎 Rule-based processing Riphah-OOP.docx

rule and annotated comparison

In [21]:
import os
import importlib.util
import json
from sklearn.metrics import classification_report, precision_recall_fscore_support

# ==========================
# STEP 1: Load predictions (rule-based)
# ==========================
pred_file = "/content/rule_based_results.json"
with open(pred_file, "r", encoding="utf-8") as f:
    pred_data = json.load(f)

# ==========================
# STEP 2: Load gold training data
# ==========================
def load_spacy_train_data(py_file):
    spec = importlib.util.spec_from_file_location("spacy_train_data", py_file)
    spacy_data = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(spacy_data)
    return spacy_data.TRAIN_DATA

# ==========================
# STEP 3: Helpers
# ==========================
def normalize_label(label):
    if label in ["OTHER", "OTHERS"]:
        return "OTHER"
    return label

def span_iou(span1, span2):
    """IoU over character spans"""
    start1, end1 = span1
    start2, end2 = span2
    inter_start = max(start1, start2)
    inter_end = min(end1, end2)
    intersection = max(0, inter_end - inter_start)
    union = max(end1, end2) - min(start1, start2)
    return intersection / union if union > 0 else 0

def evaluate_one_doc(gold_data, pred_file_data, iou_thresh=0.5):
    gold_labels, pred_labels = [], []

    # Convert preds into spans
    pred_spans = []
    for para in pred_file_data:
        para_text = para["paragraph"]
        for ent in para["entities"]:
            ent_text = ent["text"]
            label = normalize_label(ent["label"])
            start = para_text.lower().find(ent_text.lower())
            if start != -1:
                pred_spans.append((start, start+len(ent_text), label))

    # Convert gold into spans
    gold_spans = [(start, end, normalize_label(label))
                  for text, ann in gold_data
                  for start, end, label in ann["entities"]]

    matched_pred = set()

    # Gold vs preds
    for g_start, g_end, g_label in gold_spans:
        gold_labels.append(g_label)
        found_match = False
        for idx, (p_start, p_end, p_label) in enumerate(pred_spans):
            if p_label == g_label and span_iou((g_start,g_end),(p_start,p_end)) >= iou_thresh:
                pred_labels.append(p_label)
                matched_pred.add(idx)
                found_match = True
                break
        if not found_match:
            pred_labels.append("MISS")

    # False positives
    for idx, (p_start, p_end, p_label) in enumerate(pred_spans):
        if idx not in matched_pred:
            gold_labels.append("NONE")
            pred_labels.append(p_label)

    return gold_labels, pred_labels

# ==========================
# STEP 4: Loop through folders
# ==========================
root_folder = "/content/data/OOP Annotated"

all_gold, all_pred = [], []
per_folder_results = {}

for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file == "spacy_train_data.py":
            gold_file = os.path.join(root, file)
            gold_data = load_spacy_train_data(gold_file)

            folder_name = os.path.basename(root)
            docx_name = f"OOP-{folder_name}.docx"

            pred_file_data = pred_data.get(docx_name, [])

            g_labels, p_labels = evaluate_one_doc(gold_data, pred_file_data)

            all_gold.extend(g_labels)
            all_pred.extend(p_labels)

            if g_labels:
                labels = sorted(set([l for l in g_labels if l not in ["MISS","NONE"]]))
                if labels:
                    report = classification_report(
                        g_labels, p_labels,
                        labels=labels,
                        digits=4,
                        output_dict=True
                    )
                    per_folder_results[folder_name] = report["weighted avg"]

# ==========================
# STEP 5: Global Evaluation
# ==========================
print("📊 Global Classification Report (IoU-based, ignoring MISS/NONE)")
labels = sorted(set([l for l in all_gold if l not in ["MISS","NONE"]]))
print(classification_report(all_gold, all_pred, labels=labels, digits=4))

precision, recall, f1, _ = precision_recall_fscore_support(
    all_gold, all_pred, labels=labels, average="weighted"
)

print(f"\nOverall Precision: {precision:.4f}")
print(f"Overall Recall:    {recall:.4f}")
print(f"Overall F1-score:  {f1:.4f}")

# ==========================
# STEP 6: Per-folder summary
# ==========================
print("\n📂 Per-folder weighted F1-scores:")
for folder, scores in per_folder_results.items():
    print(f"{folder:30s}  P={scores['precision']:.4f}  R={scores['recall']:.4f}  F1={scores['f1-score']:.4f}")


📊 Global Classification Report (IoU-based, ignoring MISS/NONE)
              precision    recall  f1-score   support

     CONCEPT     0.5991    0.2876    0.3886       452
      METHOD     0.0000    0.0000    0.0000         1
       OTHER     0.5957    0.8485    0.7000        33
   TECHNIQUE     0.5000    1.0000    0.6667         2
       TOPIC     0.8182    0.2687    0.4045        67

   micro avg     0.6138    0.3207    0.4213       555
   macro avg     0.5026    0.4810    0.4320       555
weighted avg     0.6239    0.3207    0.4094       555


Overall Precision: 0.6239
Overall Recall:    0.3207
Overall F1-score:  0.4094

📂 Per-folder weighted F1-scores:
VU                              P=0.5928  R=0.9767  F1=0.7347
UET                             P=0.0000  R=0.0000  F1=0.0000
Riphah                          P=0.0000  R=0.0000  F1=0.0000
Bahria                          P=0.5962  R=1.0000  F1=0.7375
UMT                             P=0.0000  R=0.0000  F1=0.0000
Adelaide                 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

delete folder

In [4]:
import shutil

# Replace 'folder_name' with your folder path
shutil.rmtree('/content/data')
